In [1]:
import kagglehub, os
import os.path
import pathlib

os.environ['KAGGLE_CACHE_DIR'] = r'E:\.cache\kagglehub' 
os.environ['XDG_CACHE_HOME'] = r'E:\.cache' 

path = kagglehub.dataset_download("javaidahmadwani/lc25000")
print("Dataset downloaded to:", path)

for root, dirs, files in os.walk(path):
    print(root)
    break  


E:\AI\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset downloaded to: C:\Users\aksha\.cache\kagglehub\datasets\javaidahmadwani\lc25000\versions\1
C:\Users\aksha\.cache\kagglehub\datasets\javaidahmadwani\lc25000\versions\1


In [2]:
data_dir = os.path.join(path, "lung_colon_image_set")


In [3]:
import pathlib
data_dir = pathlib.Path(data_dir)
print("Data folder:", data_dir)



Data folder: C:\Users\aksha\.cache\kagglehub\datasets\javaidahmadwani\lc25000\versions\1\lung_colon_image_set


In [4]:
import os, pathlib

data_dir = pathlib.Path(path) / "lung_colon_image_set"

for root, dirs, files in os.walk(data_dir):
    print("📂", root)
    for d in dirs:
        print("  └──", d)
    break 


📂 C:\Users\aksha\.cache\kagglehub\datasets\javaidahmadwani\lc25000\versions\1\lung_colon_image_set
  └── Test Set
  └── Train and Validation Set


In [5]:
class_names = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
print("Detected Classes:", class_names)

all_paths, all_labels = [], []
for cls in class_names:
    label = class_names.index(cls)
    for img in (data_dir/cls).glob("*.jpeg"):
        all_paths.append(str(img))
        all_labels.append(label)

print("Total images found:", len(all_paths))


Detected Classes: ['Test Set', 'Train and Validation Set']
Total images found: 0


In [6]:
import os, pathlib, random
from collections import defaultdict
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf

base_path = pathlib.Path(path) / "lung_colon_image_set"

print("Top-level directory:", base_path)
print("Top-level contents:")
for p in sorted(base_path.iterdir()):
    print(" ", p.name)

names = {p.name.lower(): p for p in base_path.iterdir() if p.is_dir()}
trainval_folder = None
test_folder = None
for k,v in names.items():
    if "train" in k and "val" in k or "train" in k and "validation" in k:
        trainval_folder = v
    if "test" in k:
        test_folder = v

if trainval_folder is None or test_folder is None:
    # try more flexible match
    for p in base_path.iterdir():
        if p.is_dir():
            name = p.name.lower()
            if "test" in name and test_folder is None:
                test_folder = p
            elif ("train" in name or "validation" in name or "val" in name) and trainval_folder is None:
                trainval_folder = p

if trainval_folder is None or test_folder is None:
    raise RuntimeError("Could not automatically find Train+Validation and Test folders. "
                       "Please verify the folder structure under:\n  " + str(base_path))

print("\nUsing:")
print(" Train+Val folder:", trainval_folder)
print(" Test folder      :", test_folder)

def collect_class_files(parent_dir):
    parent_dir = pathlib.Path(parent_dir)
    class_dirs = [p for p in parent_dir.iterdir() if p.is_dir()]
    
    if not class_dirs:
        # collect images in parent_dir
        exts = ("*.jpg","*.jpeg","*.png","*.tif","*.tiff")
        files = []
        for e in exts:
            files += list(parent_dir.glob(e))
        return {parent_dir.name: [str(p) for p in files]}
    d = {}
    for c in class_dirs:
        files = []
        for ext in ("*.jpg","*.jpeg","*.png","*.tif","*.tiff"):
            files.extend(list(c.glob(ext)))
        # if none found in this candidate, try deeper one level (rare)
        if len(files)==0:
            for child in c.iterdir():
                if child.is_dir():
                    for ext in ("*.jpg","*.jpeg","*.png","*.tif","*.tiff"):
                        files.extend(list(child.glob(ext)))
        d[c.name] = [str(p) for p in sorted(files)]
    return d

trainval_dict = collect_class_files(trainval_folder)
test_dict = collect_class_files(test_folder)

print("\nClasses found in Train+Val:", sorted(trainval_dict.keys()))
print("Classes found in Test     :", sorted(test_dict.keys()))

trainval_classes = sorted(trainval_dict.keys())
test_classes = sorted(test_dict.keys())
if set(trainval_classes) != set(test_classes):
    print("\nWARNING: class sets differ between Train+Val and Test. We'll union them and map accordingly.")
all_classes = sorted(list(set(trainval_classes).union(set(test_classes))))
print("Final class list:", all_classes)

def build_lists_from_dict(dct, class_list):
    paths, labels = [], []
    for cls in class_list:
        fl = dct.get(cls, [])
        for f in fl:
            paths.append(f)
            labels.append(class_list.index(cls))
    return paths, labels

test_paths, test_labels = build_lists_from_dict(test_dict, all_classes)
trainval_paths, trainval_labels = build_lists_from_dict(trainval_dict, all_classes)

print(f"\nFound {len(trainval_paths)} images in Train+Val and {len(test_paths)} images in Test.")

if len(trainval_paths)==0:
    print("Train+Val seems empty — doing global recursive scan and manual split.")
    exts = ("*.jpg","*.jpeg","*.png","*.tif","*.tiff")
    global_files = []
    for ext in exts:
        global_files += list(base_path.rglob(ext))
    if len(global_files)==0:
        raise RuntimeError("No image files found anywhere under dataset folder.")
    # attempt to infer labels from parent folder names
    gp = [str(p) for p in global_files]
    inferred_labels = [pathlib.Path(p).parent.name for p in gp]
    classes = sorted(list(set(inferred_labels)))
    paths = gp
    labels = [classes.index(x) for x in inferred_labels]
    paths_tmp, test_paths, labels_tmp, test_labels = train_test_split(paths, labels, test_size=0.2, stratify=labels, random_state=42)
    train_paths, val_paths, train_labels, val_labels = train_test_split(paths_tmp, labels_tmp, test_size=0.125, stratify=labels_tmp, random_state=42)
    all_classes = classes
else:
    stratify = trainval_labels if len(all_classes) > 1 else None
    if len(trainval_paths) > 0:
        train_paths, val_paths, train_labels, val_labels = train_test_split(
            trainval_paths, trainval_labels, test_size=0.125, stratify=stratify, random_state=42)
    else:
        raise RuntimeError("No images found in Train+Val directory; cannot split.")

from collections import Counter
print("\nFinal dataset sizes:")
print(" Train:", len(train_paths))
print(" Val  :", len(val_paths))
print(" Test :", len(test_paths))

def counts_per_class(paths, labels, cls_list):
    cnt = Counter()
    for p,l in zip(paths,labels):
        cnt[cls_list[l]] += 1
    return cnt

print("\nTrain class counts:", counts_per_class(train_paths, train_labels, all_classes))
print("Val   class counts:", counts_per_class(val_paths, val_labels, all_classes))
print("Test  class counts:", counts_per_class(test_paths, test_labels, all_classes))

IMG_SIZE = 256
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

def preprocess_fn(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    if augment:
        # simple augmentations
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_flip_up_down(img)
        img = tf.image.random_brightness(img, max_delta=0.08)
        img = tf.image.random_contrast(img, 0.9, 1.1)
    return img, label

def make_ds(paths, labels, batch, shuffle=False, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(10000, len(paths)))
    ds = ds.map(lambda p,l: preprocess_fn(p,l,augment=augment), num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.batch(batch).prefetch(AUTOTUNE)
    else:
        ds = ds.batch(batch).prefetch(AUTOTUNE)
    return ds

train_ds = make_ds(train_paths, train_labels, BATCH_SIZE, shuffle=True, augment=True)
val_ds   = make_ds(val_paths, val_labels, BATCH_SIZE, shuffle=False, augment=False)
test_ds  = make_ds(test_paths, test_labels, BATCH_SIZE, shuffle=False, augment=False)

print("\nTF datasets ready. Examples from one batch:")
for images, labels in train_ds.take(1):
    print(" batch images shape:", images.shape)
    print(" batch labels shape:", labels.shape)
    break

import pickle
with open("lc25000_splits.pkl","wb") as f:
    pickle.dump({
        "classes": all_classes,
        "train_paths": train_paths,
        "train_labels": train_labels,
        "val_paths": val_paths,
        "val_labels": val_labels,
        "test_paths": test_paths,
        "test_labels": test_labels
    }, f)

print("\nSaved split metadata to lc25000_splits.pkl")


Top-level directory: C:\Users\aksha\.cache\kagglehub\datasets\javaidahmadwani\lc25000\versions\1\lung_colon_image_set
Top-level contents:
  Test Set
  Train and Validation Set

Using:
 Train+Val folder: C:\Users\aksha\.cache\kagglehub\datasets\javaidahmadwani\lc25000\versions\1\lung_colon_image_set\Train and Validation Set
 Test folder      : C:\Users\aksha\.cache\kagglehub\datasets\javaidahmadwani\lc25000\versions\1\lung_colon_image_set\Test Set

Classes found in Train+Val: ['colon_aca', 'colon_n', 'lung_aca', 'lung_n', 'lung_scc']
Classes found in Test     : ['colon_aca', 'colon_n', 'lung_aca', 'lung_n', 'lung_scc']
Final class list: ['colon_aca', 'colon_n', 'lung_aca', 'lung_n', 'lung_scc']

Found 22501 images in Train+Val and 2499 images in Test.

Final dataset sizes:
 Train: 19688
 Val  : 2813
 Test : 2499

Train class counts: Counter({'lung_scc': 3938, 'colon_n': 3938, 'lung_n': 3938, 'lung_aca': 3937, 'colon_aca': 3937})
Val   class counts: Counter({'lung_aca': 563, 'colon_aca':

In [7]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import numpy as np
import pickle

splits = pickle.load(open("lc25000_splits.pkl","rb"))
NUM_CLASSES = len(splits['classes'])
IMG_SIZE = 256

class StatisticalWaveAttention(layers.Layer):
    def __init__(self, mode='pearson', a=3.1, h=1.065, c_param=1.1, k=0.1, **kwargs):
        super().__init__(**kwargs)
        assert mode in ('pearson', 'spearman')
        self.mode = mode
        self.a = a
        self.h = h
        self.c_param = c_param
        self.k = k
        self.conv1d = None
        self.bn = None
        self.sig = None

    def build(self, input_shape):
        self.H = input_shape[1]
        self.W = input_shape[2]
        self.C = int(input_shape[3])
        # Conv1D to process sequence across spatial dimension
        self.conv1d = layers.Conv1D(filters=self.C, kernel_size=3, padding='same', activation='relu')
        self.bn = layers.BatchNormalization()
        self.sig = layers.Activation('sigmoid')
        super().build(input_shape)

    def rank_transform(self, x):
        # x: [B, HW]
        order = tf.argsort(x, axis=1)
        ranks = tf.argsort(order, axis=1)
        ranks = tf.cast(ranks, tf.float32)
        denom = tf.cast(tf.shape(x)[1] - 1, tf.float32)
        denom = tf.where(denom <= 0.0, 1.0, denom)
        ranks = ranks / denom
        return ranks

    def compute_pcc_cos(self, x):
        B = tf.shape(x)[0]
        H = tf.shape(x)[1]
        W = tf.shape(x)[2]
        C = tf.shape(x)[3]
        HW = H * W

        x_flat = tf.reshape(x, [B, HW, C])  # [B, HW, C]
        mu = tf.reduce_mean(x_flat, axis=2)  # [B, HW]

        mu_mean = tf.reduce_mean(mu, axis=1, keepdims=True)  # [B,1]
        mu_c = mu - mu_mean  # [B,HW]

        x_mean = tf.reduce_mean(x_flat, axis=1, keepdims=True)  # [B,1,C]
        x_c = x_flat - x_mean  # [B,HW,C]

        num = tf.reduce_sum(tf.expand_dims(mu_c, -1) * x_c, axis=1)  # [B,C]
        mu_sq = tf.reduce_sum(tf.square(mu_c), axis=1, keepdims=True)
        x_sq = tf.reduce_sum(tf.square(x_c), axis=1)
        denom = tf.sqrt(tf.maximum(mu_sq * x_sq, 1e-12))
        pcc = num / (denom + 1e-12)

        mu_norm = tf.norm(mu, axis=1, keepdims=True)
        x_norms = tf.norm(x_flat, axis=1)
        cos = tf.reduce_sum(tf.expand_dims(mu, -1) * x_flat, axis=1) / ((mu_norm * x_norms) + 1e-12)

        pcc = tf.clip_by_value(pcc, -1.0, 1.0)
        cos = tf.clip_by_value(cos, -1.0, 1.0)
        return pcc, cos

    def call(self, inputs, training=None):
        B = tf.shape(inputs)[0]
        H = tf.shape(inputs)[1]
        W = tf.shape(inputs)[2]
        C = tf.shape(inputs)[3]
        HW = H * W

        if self.mode == 'pearson':
            pcc, cos = self.compute_pcc_cos(inputs)
            fss = 0.5 * (pcc + cos)
        else:
            x_flat = tf.reshape(inputs, [B, HW, C])
            mu = tf.reduce_mean(x_flat, axis=2)
            mu_ranks = self.rank_transform(mu)
            x_trans = tf.transpose(x_flat, perm=[0,2,1])  # [B,C,HW]
            x_rows = tf.reshape(x_trans, [-1, HW])        # [B*C, HW]
            x_ranks_rows = self.rank_transform(x_rows)    # [B*C, HW]
            x_ranks = tf.reshape(x_ranks_rows, [B, C, HW])
            x_ranks = tf.transpose(x_ranks, perm=[0,2,1]) # [B,HW,C]

            mu_mean = tf.reduce_mean(mu_ranks, axis=1, keepdims=True)
            mu_c = mu_ranks - mu_mean
            x_mean = tf.reduce_mean(x_ranks, axis=1, keepdims=True)
            x_c = x_ranks - x_mean
            num = tf.reduce_sum(tf.expand_dims(mu_c, -1) * x_c, axis=1)
            mu_sq = tf.reduce_sum(tf.square(mu_c), axis=1, keepdims=True)
            x_sq = tf.reduce_sum(tf.square(x_c), axis=1)
            denom = tf.sqrt(tf.maximum(mu_sq * x_sq, 1e-12))
            pcc_ranks = num / (denom + 1e-12)

            mu_norm = tf.norm(mu_ranks, axis=1, keepdims=True)
            x_norms = tf.norm(x_ranks, axis=1)
            cos_ranks = tf.reduce_sum(tf.expand_dims(mu_ranks, -1) * x_ranks, axis=1) / ((mu_norm * x_norms) + 1e-12)

            pcc = tf.clip_by_value(pcc_ranks, -1.0, 1.0)
            cos = tf.clip_by_value(cos_ranks, -1.0, 1.0)
            fss = 0.5 * (pcc + cos)

        scores = tf.exp(self.a * (fss - self.h)) * self.c_param + self.k
        scores = tf.clip_by_value(scores, 0.0, 1.0)
        weights = tf.reshape(scores, [-1, 1, 1, tf.shape(scores)[1]])

        x_weighted = inputs * weights
        seq = tf.reshape(x_weighted, [-1, HW, inputs.shape[-1]])
        conv_out = self.conv1d(seq)
        conv_out = self.bn(conv_out, training=training)
        conv_out = self.sig(conv_out)
        conv_out = tf.reshape(conv_out, [-1, H, W, inputs.shape[-1]])
        out = inputs * conv_out
        return out

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"mode": self.mode, "a": self.a, "h": self.h, "c_param": self.c_param, "k": self.k})
        return cfg

def build_sawl_net(img_size=IMG_SIZE, num_classes=NUM_CLASSES, use_pcc=True, use_srcc=True):
    inputs = layers.Input(shape=(img_size, img_size, 3))
    base = tf.keras.applications.MobileNetV2(input_tensor=inputs, include_top=False, weights='imagenet')

    outer_name = 'block_1_expand_relu'
    inner_name = 'block_13_expand_relu'
    outer_feat = base.get_layer(outer_name).output
    inner_feat = base.get_layer(inner_name).output
    trunk = base.output

    outer_up = None
    inner_up = None
    if use_pcc:
        outer_att = StatisticalWaveAttention(mode='pearson', name='pcc_att')(outer_feat)
        outer_up = layers.Conv2D(trunk.shape[-1], kernel_size=1, padding='same', activation='relu')(outer_att)
        outer_up = layers.Resizing(trunk.shape[1], trunk.shape[2], interpolation='bilinear')(outer_up)

    if use_srcc:
        inner_att = StatisticalWaveAttention(mode='spearman', name='srcc_att')(inner_feat)
        inner_up = layers.Conv2D(trunk.shape[-1], kernel_size=1, padding='same', activation='relu')(inner_att)
        inner_up = layers.Resizing(trunk.shape[1], trunk.shape[2], interpolation='bilinear')(inner_up)

    merged = [trunk]
    if outer_up is not None:
        merged.append(outer_up)
    if inner_up is not None:
        merged.append(inner_up)

    if len(merged) > 1:
        x = layers.Concatenate()(merged)
        x = layers.Conv2D(512, kernel_size=3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
    else:
        x = trunk

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name=f"SAWLNet_pcc_{use_pcc}_srcc_{use_srcc}")
    return model

model = build_sawl_net()
model.summary()


Model: "SAWLNet_pcc_True_srcc_True"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 256, 256, 3  0           []                               
                                )]                                                                
                                                                                                  
 Conv1 (Conv2D)                 (None, 128, 128, 32  864         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 bn_Conv1 (BatchNormalization)  (None, 128, 128, 32  128         ['Conv1[0][0]']                  
                                )                                        

In [8]:
import tensorflow as tf
try:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("Mixed precision enabled:", tf.keras.mixed_precision.global_policy())
except Exception as e:
    print("Mixed precision not enabled:", e)


Your GPUs may run slowly with dtype policy mixed_float16 because they do not have compute capability of at least 7.0. Your GPUs:
  DML, no compute capability (probably not an Nvidia GPU) (x2)
See https://developer.nvidia.com/cuda-gpus for a list of GPUs and their compute capabilities.
If you will use compatible GPU(s) not attached to this host, e.g. by running a multi-worker model, you can ignore this warning. This message will only be logged once
Mixed precision enabled: <Policy "mixed_float16">


In [9]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import pickle, numpy as np, os

splits = pickle.load(open("lc25000_splits.pkl","rb"))
NUM_CLASSES = len(splits['classes'])
IMG_SIZE = 256

class StatisticalWaveAttention(layers.Layer):
    def __init__(self, mode='pearson', a=3.1, h=1.065, c_param=1.1, k=0.1, **kwargs):
        super().__init__(**kwargs)
        assert mode in ('pearson', 'spearman')
        self.mode = mode
        self.a = a
        self.h = h
        self.c_param = c_param
        self.k = k
        self.conv1d = None
        self.bn = None
        self.sig = None

    def build(self, input_shape):
        self.H = input_shape[1]
        self.W = input_shape[2]
        self.C = int(input_shape[3])
        self.conv1d = layers.Conv1D(filters=self.C, kernel_size=3, padding='same', activation='relu', dtype=self.compute_dtype)
        self.bn = layers.BatchNormalization(dtype=self.compute_dtype)
        self.sig = layers.Activation('sigmoid', dtype=self.compute_dtype)
        super().build(input_shape)

    def rank_transform(self, x):
        order = tf.argsort(x, axis=1)
        ranks = tf.argsort(order, axis=1)
        ranks = tf.cast(ranks, tf.float32) 
        denom = tf.cast(tf.shape(x)[1] - 1, tf.float32)
        denom = tf.where(denom <= 0.0, 1.0, denom)
        ranks = ranks / denom
        return tf.cast(ranks, x.dtype)

    def compute_pcc_cos(self, x):
        orig_dtype = x.dtype
        x32 = tf.cast(x, tf.float32)
        B = tf.shape(x32)[0]; H = tf.shape(x32)[1]; W = tf.shape(x32)[2]; C = tf.shape(x32)[3]
        HW = H * W
        x_flat = tf.reshape(x32, [B, HW, C])  
        mu = tf.reduce_mean(x_flat, axis=2)
        mu_mean = tf.reduce_mean(mu, axis=1, keepdims=True) 
        mu_c = mu - mu_mean

        x_mean = tf.reduce_mean(x_flat, axis=1, keepdims=True)
        x_c = x_flat - x_mean

        num = tf.reduce_sum(tf.expand_dims(mu_c, -1) * x_c, axis=1)
        mu_sq = tf.reduce_sum(tf.square(mu_c), axis=1, keepdims=True)
        x_sq = tf.reduce_sum(tf.square(x_c), axis=1)
        denom = tf.sqrt(tf.maximum(mu_sq * x_sq, 1e-12))
        pcc = num / (denom + 1e-12)

        mu_norm = tf.norm(mu, axis=1, keepdims=True)
        x_norms = tf.norm(x_flat, axis=1)
        cos = tf.reduce_sum(tf.expand_dims(mu, -1) * x_flat, axis=1) / ((mu_norm * x_norms) + 1e-12)

        pcc = tf.clip_by_value(pcc, -1.0, 1.0)
        cos = tf.clip_by_value(cos, -1.0, 1.0)

        return tf.cast(pcc, orig_dtype), tf.cast(cos, orig_dtype)

    def call(self, inputs, training=None):
        in_dtype = inputs.dtype
        B = tf.shape(inputs)[0]; H = tf.shape(inputs)[1]; W = tf.shape(inputs)[2]; C = tf.shape(inputs)[3]
        HW = H * W

        if self.mode == 'pearson':
            pcc, cos = self.compute_pcc_cos(inputs)
            fss = 0.5 * (pcc + cos)  # dtype = in_dtype
        else:
            x_flat = tf.reshape(tf.cast(inputs, tf.float32), [B, HW, C])
            mu = tf.reduce_mean(x_flat, axis=2)  # [B,HW]
            mu_ranks = self.rank_transform(tf.cast(mu, in_dtype))
            x_trans = tf.transpose(tf.cast(x_flat, in_dtype), perm=[0,2,1])  # [B,C,HW]
            x_rows = tf.reshape(x_trans, [-1, HW])  # [B*C, HW]
            x_ranks_rows = self.rank_transform(x_rows)
            x_ranks = tf.reshape(x_ranks_rows, [B, C, HW])
            x_ranks = tf.transpose(x_ranks, perm=[0,2,1])  # [B,HW,C]

            mu32 = tf.cast(mu_ranks, tf.float32)
            x32 = tf.cast(x_ranks, tf.float32)
            mu_mean = tf.reduce_mean(mu32, axis=1, keepdims=True)
            mu_c = mu32 - mu_mean
            x_mean = tf.reduce_mean(x32, axis=1, keepdims=True)
            x_c = x32 - x_mean
            num = tf.reduce_sum(tf.expand_dims(mu_c, -1) * x_c, axis=1)
            mu_sq = tf.reduce_sum(tf.square(mu_c), axis=1, keepdims=True)
            x_sq = tf.reduce_sum(tf.square(x_c), axis=1)
            denom = tf.sqrt(tf.maximum(mu_sq * x_sq, 1e-12))
            pcc_ranks = num / (denom + 1e-12)
            mu_norm = tf.norm(mu32, axis=1, keepdims=True)
            x_norms = tf.norm(x32, axis=1)
            cos_ranks = tf.reduce_sum(tf.expand_dims(mu32, -1) * x32, axis=1) / ((mu_norm * x_norms) + 1e-12)
            pcc = tf.cast(pcc_ranks, in_dtype)
            cos = tf.cast(cos_ranks, in_dtype)
            fss = 0.5 * (pcc + cos)

        fss32 = tf.cast(fss, tf.float32)
        scores32 = tf.exp(self.a * (fss32 - self.h)) * self.c_param + self.k 
        scores = tf.clip_by_value(scores32, 0.0, 1.0)
        scores = tf.cast(scores, in_dtype)  
        weights = tf.reshape(scores, [-1, 1, 1, tf.shape(scores)[1]]) 

        x_weighted = inputs * weights

        seq = tf.reshape(x_weighted, [-1, HW, inputs.shape[-1]]) 
        conv_out = self.conv1d(seq)
        conv_out = self.bn(conv_out, training=training)
        conv_out = self.sig(conv_out)
        conv_out = tf.reshape(conv_out, [-1, H, W, inputs.shape[-1]])
        out = inputs * conv_out
        return out

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"mode": self.mode, "a": self.a, "h": self.h, "c_param": self.c_param, "k": self.k})
        return cfg

def build_sawl_net_fixed(img_size=IMG_SIZE, num_classes=NUM_CLASSES, use_pcc=True, use_srcc=True):
    inputs = layers.Input(shape=(img_size, img_size, 3))
    base = tf.keras.applications.MobileNetV2(input_tensor=inputs, include_top=False, weights='imagenet')

    outer_name = 'block_1_expand_relu'
    inner_name = 'block_13_expand_relu'
    outer_feat = base.get_layer(outer_name).output
    inner_feat = base.get_layer(inner_name).output
    trunk = base.output

    outer_up = None
    inner_up = None
    if use_pcc:
        outer_att = StatisticalWaveAttention(mode='pearson', name='pcc_att')(outer_feat)
        outer_up = layers.Conv2D(trunk.shape[-1], kernel_size=1, padding='same', activation='relu')(outer_att)
        outer_up = layers.Resizing(trunk.shape[1], trunk.shape[2], interpolation='bilinear')(outer_up)

    if use_srcc:
        inner_att = StatisticalWaveAttention(mode='spearman', name='srcc_att')(inner_feat)
        inner_up = layers.Conv2D(trunk.shape[-1], kernel_size=1, padding='same', activation='relu')(inner_att)
        inner_up = layers.Resizing(trunk.shape[1], trunk.shape[2], interpolation='bilinear')(inner_up)

    merged = [trunk]
    if outer_up is not None:
        merged.append(outer_up)
    if inner_up is not None:
        merged.append(inner_up)

    if len(merged) > 1:
        x = layers.Concatenate()(merged)
        x = layers.Conv2D(512, kernel_size=3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
    else:
        x = trunk

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32')(x)  # ensure output in float32 for losses

    model = Model(inputs, outputs, name=f"SAWLNet_fixed_pcc_{use_pcc}_srcc_{use_srcc}")
    return model

from tensorflow.keras import layers
model_fixed = build_sawl_net_fixed()
model_fixed.summary()

for imgs, labs in train_ds.take(1):
    imgs_small = imgs  
    _ = model_fixed(imgs_small, training=False)
    print("Forward pass OK, output shape:", _.shape)
    break

build_sawl_net = build_sawl_net_fixed
model = model_fixed
print("Rebuilt SAWL-Net with dtype-safe attention layer.")


Model: "SAWLNet_fixed_pcc_True_srcc_True"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_2 (InputLayer)           [(None, 256, 256, 3  0           []                               
                                )]                                                                
                                                                                                  
 Conv1 (Conv2D)                 (None, 128, 128, 32  864         ['input_2[0][0]']                
                                )                                                                 
                                                                                                  
 bn_Conv1 (BatchNormalization)  (None, 128, 128, 32  128         ['Conv1[0][0]']                  
                                )                                  

In [10]:
import tensorflow as tf
policy = tf.keras.mixed_precision.global_policy()
print("Current Keras policy:", policy)

Current Keras policy: <Policy "mixed_float16">


In [11]:
import os, tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping, CSVLogger

print("Clearing session and forcing float32 policy...")
K.clear_session()
try:
    tf.keras.mixed_precision.set_global_policy('float32')
except Exception as e:
    print("Could not set mixed precision policy (non-fatal):", e)
print("Current policy:", tf.keras.mixed_precision.global_policy())

try:
    model = build_sawl_net(img_size=IMG_SIZE, num_classes=NUM_CLASSES, use_pcc=True, use_srcc=True)
except Exception as e:
    print("Error building model:", e)
    raise

LR = 1e-5
opt = tf.keras.optimizers.Adam(learning_rate=LR)
model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

os.makedirs("models", exist_ok=True)
checkpoint_path = "models/sawl_best_full.h5"
callbacks = [
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, verbose=1, min_lr=1e-7),
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    CSVLogger("models/sawl_full_training_log.csv")
]

if os.path.exists(checkpoint_path):
    try:
        print("Found existing checkpoint. Loading weights and continuing training from that checkpoint...")
        model.load_weights(checkpoint_path)
        print("Weights loaded from", checkpoint_path)
    except Exception as e:
        print("Could not load checkpoint weights directly (will continue training from scratch). Error:", e)

EPOCHS = 20
print("Starting training for", EPOCHS, "epochs (this may take a while).")
history = model.fit(train_ds,
                    validation_data=val_ds,
                    epochs=EPOCHS,
                    callbacks=callbacks)

model.save("models/sawl_full_saved_model_float32", include_optimizer=True)
print("Training done. Best checkpoint saved to:", checkpoint_path)
print("SavedModel exported to: models/sawl_full_saved_model_float32")


Clearing session and forcing float32 policy...
Current policy: <Policy "float32">
Model: "SAWLNet_fixed_pcc_True_srcc_True"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 256, 256, 3  0           []                               
                                )]                                                                
                                                                                                  
 Conv1 (Conv2D)                 (None, 128, 128, 32  864         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 bn_Conv1 (BatchNormalization)  (None, 128, 128, 32  128         ['Conv1[0][0]']    

KeyboardInterrupt: 

In [12]:
import pickle, numpy as np, tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

custom_objects = {"StatisticalWaveAttention": StatisticalWaveAttention}

best_path = "models/sawl_best_full.h5"
if os.path.exists(best_path):
    try:
        best_model = tf.keras.models.load_model(best_path, custom_objects=custom_objects, compile=False)
        print("Loaded model from", best_path)
    except Exception as e:
        print("Error loading .h5 model; trying SavedModel folder. Error:", e)
        best_model = tf.keras.models.load_model("models/sawl_full_saved_model", custom_objects=custom_objects, compile=False)
        print("Loaded SavedModel.")
else:
    print("No .h5 checkpoint found, using in-memory model variable 'model' (if present).")
    try:
    except NameError:
        raise RuntimeError("No model available in memory; run training cell first.")
    best_model = model

y_true, y_pred, y_prob = [], [], []
for imgs, labs in test_ds:
    preds = best_model.predict(imgs)
    y_prob.append(preds)
    y_pred.append(np.argmax(preds, axis=1))
    y_true.append(labs.numpy())
y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)
y_prob = np.concatenate(y_prob)

splits = pickle.load(open("lc25000_splits.pkl","rb"))
class_names = splits['classes']
print("Classification report:")
print(classification_report(y_true, y_pred, target_names=class_names))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

try:
    from sklearn.preprocessing import label_binarize
    y_true_bin = label_binarize(y_true, classes=list(range(len(class_names))))
    roc_macro = roc_auc_score(y_true_bin, y_prob, average='macro', multi_class='ovr')
    print("ROC-AUC (macro OVR):", roc_macro)
except Exception as e:
    print("ROC-AUC calculation failed:", e)


Loaded model from models/sawl_best_full.h5
1/1 [==============================] - 1s 819ms/step
Classification report:
              precision    recall  f1-score   support

   colon_aca       1.00      1.00      1.00       500
     colon_n       1.00      1.00      1.00       500
    lung_aca       1.00      1.00      1.00       500
      lung_n       1.00      1.00      1.00       500
    lung_scc       1.00      1.00      1.00       499

    accuracy                           1.00      2499
   macro avg       1.00      1.00      1.00      2499
weighted avg       1.00      1.00      1.00      2499

Confusion matrix:
 [[500   0   0   0   0]
 [  0 500   0   0   0]
 [  0   0 498   0   2]
 [  0   0   0 500   0]
 [  0   0   0   0 499]]
ROC-AUC (macro OVR): 0.9999893886823171


In [ ]:
import tensorflow as tf, matplotlib.pyplot as plt, numpy as np

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        loss = predictions[:, pred_index]
    grads = tape.gradient(loss, conv_outputs)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    conv = conv_outputs[0]
    heatmap = tf.reduce_sum(conv * pooled, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-12)
    heatmap = tf.image.resize(heatmap[..., tf.newaxis], (IMG_SIZE, IMG_SIZE))
    return tf.squeeze(heatmap).numpy()

for imgs, labs in test_ds.take(1):
    sample_img = imgs[0:1]
    sample_label = int(labs[0].numpy())
    break

last_conv = None
for layer in reversed(best_model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv = layer.name
        break
print("Using conv layer:", last_conv)

heatmap = make_gradcam_heatmap(sample_img, best_model, last_conv)
plt.figure(figsize=(10,5))
plt.subplot(1,2,1); plt.imshow(sample_img[0]); plt.title(f"True: {class_names[sample_label]}"); plt.axis('off')
plt.subplot(1,2,2); plt.imshow(sample_img[0]); plt.imshow(heatmap, cmap='jet', alpha=0.4); plt.title("Grad-CAM"); plt.axis('off')
plt.show()


In [ ]:
from sklearn.manifold import TSNE
import numpy as np, matplotlib.pyplot as plt, tensorflow as tf

feat_model = tf.keras.Model(inputs=best_model.input, outputs=best_model.layers[-2].output)

N = 1000
imgs_list, labs_list = [], []
count = 0
for imgs, labs in test_ds:
    for i in range(imgs.shape[0]):
        imgs_list.append(imgs[i].numpy()); labs_list.append(int(labs[i].numpy()))
        count += 1
        if count >= N: break
    if count >= N: break

embs = feat_model.predict(np.array(imgs_list), batch_size=32)
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init='pca')
emb2 = tsne.fit_transform(embs)
plt.figure(figsize=(8,6))
for idx, name in enumerate(class_names):
    mask = np.array(labs_list) == idx
    plt.scatter(emb2[mask,0], emb2[mask,1], label=name, s=8, alpha=0.7)
plt.legend(markerscale=2)
plt.title("t-SNE of penultimate embeddings (subset)")
plt.show()


In [ ]:
import numpy as np, matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

base = tf.keras.applications.MobileNetV2(input_shape=(IMG_SIZE,IMG_SIZE,3), include_top=False, weights='imagenet')
extractor = tf.keras.Model(base.input, [base.get_layer('block_1_expand_relu').output,
                                        base.get_layer('block_13_expand_relu').output])

o_feat, i_feat = extractor(sample_img)
pcc_layer = StatisticalWaveAttention(mode='pearson')
srcc_layer = StatisticalWaveAttention(mode='spearman')
o_att = pcc_layer(o_feat, training=False)
i_att = srcc_layer(i_feat, training=False)

att_map = o_att[0,:,:,0].numpy()
H, W = att_map.shape
X, Y = np.meshgrid(np.arange(W), np.arange(H))

fig = plt.figure(figsize=(12,5))
ax = fig.add_subplot(1,2,1, projection='3d')
ax.plot_surface(X, Y, att_map, cmap='viridis', linewidth=0, antialiased=False)
ax.set_title("Outer Attention Channel (surface)")
ax2 = fig.add_subplot(1,2,2)
c = ax2.contourf(att_map, cmap='viridis')
fig.colorbar(c, ax=ax2)
ax2.set_title("Outer Attention Channel (contour)")
plt.show()


In [ ]:
import pandas as pd, numpy as np, tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

variants = [
    ("baseline", False, False),
    ("pcc_only", True, False),
    ("srcc_only", False, True),
    ("sawl_full", True, True)
]

EPOCHS_AB = 60 
results = []
custom_objects = {"StatisticalWaveAttention": StatisticalWaveAttention}

for name, use_pcc, use_srcc in variants:
    print("\n=== Variant:", name, use_pcc, use_srcc, "===")
    m = build_sawl_net(img_size=IMG_SIZE, num_classes=NUM_CLASSES, use_pcc=use_pcc, use_srcc=use_srcc)
    m.compile(optimizer=tf.keras.optimizers.Adam(LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    ck = f"models/ckpt_{name}.h5"
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(ck, monitor='val_accuracy', save_best_only=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-7),
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
    ]
    m.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_AB, callbacks=cbs)
    m_best = tf.keras.models.load_model(ck, custom_objects=custom_objects, compile=False)
    ytr, ypr = [], []
    for imgs, labs in test_ds:
        p = m_best.predict(imgs)
        ypr.append(np.argmax(p, axis=1))
        ytr.append(labs.numpy())
    ytr = np.concatenate(ytr); ypr = np.concatenate(ypr)
    acc = accuracy_score(ytr, ypr)
    prec = precision_score(ytr, ypr, average='macro', zero_division=0)
    rec = recall_score(ytr, ypr, average='macro', zero_division=0)
    f1 = f1_score(ytr, ypr, average='macro', zero_division=0)
    print(f"Variant {name}: acc={acc:.4f}, prec={prec:.4f}, rec={rec:.4f}, f1={f1:.4f}")
    results.append((name, acc, prec, rec, f1))

df = pd.DataFrame(results, columns=['variant','accuracy','precision','recall','f1'])
print(df)
df.to_csv("models/ablation_results.csv", index=False)
